# 2. Can player movement alone predict the labels?

The initial labels (notebook 1) cover 82% of the frames, mostly where the ball is visible. To fill
the rest we need something that works **without the ball**. Test: describe each frame only by the
players (team shape, speeds), train LightGBM on 80% of the match, and check it on the other 20%.
Then train on all labels and predict the unlabelled frames (section 12).

The code lives in `possession/`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

import sys
sys.path.insert(0, 'evaluation')
from possession_metrics import spell_scores

from possession import features, model, rules
from possession.data import (ATHLETIC, BARCA, DEAD, FPS, HALVES, LABEL_NAME, SECOND_HALF_KICKOFF, TEAM_NAME,
                             UNLABELLED, attacking_direction, distance_from_own_goal, graded_frames, load_events,
                             load_tracking, runs_of)

## 1. Data and initial labels

Loaded from the raw files; the labels come from the same rules as notebook 1, in one call.
`trusted_dead` marks the D frames found by a rule other than R2 / R5 (notebook 1, section 6).

In [ ]:
tracking = load_tracking()
events = load_events()
players_xy, players_speed, slot_team = tracking['players_xy'], tracking['players_speed'], tracking['slot_team']
graded = graded_frames()
labels, trusted_dead = rules.initial_labels(tracking, events)
uncertain_dead = (labels == DEAD) & ~trusted_dead
print(f'D frames only from R2/R5: {uncertain_dead[graded].mean():.1%} of graded')
pd.Series(labels[graded]).map(LABEL_NAME).value_counts(normalize=True).round(3)

## 2. Player velocity towards the opponent's goal

From each slot's movement over 10 frames (0.4 s), signed by the team's attacking direction:
positive = running towards the opponent's goal. Speeds above 12 m/s are a slot jumping to
another player, so they are dropped.

In [ ]:
direction = attacking_direction(players_xy)
from_own_goal = distance_from_own_goal(players_xy, slot_team, direction)
forward = features.forward_velocity(players_xy, slot_team, direction)

FRAME = 30683  # a moment where a Barça player has the ball (notebook 1)
in_use = ~np.isnan(players_xy[FRAME, :, 0])
pd.DataFrame({'team': [TEAM_NAME[t] for t in slot_team[in_use]],
              'from own goal (m)': from_own_goal[FRAME, in_use],
              'forward speed (m/s)': forward[FRAME, in_use]}).astype({'from own goal (m)': float,
                                                                      'forward speed (m/s)': float}).round(1).head(6)

## 3. Team features

Per team (`barca_…`, `athletic_…`), per frame:

| Feature | Meaning |
|---|---|
| `players_tracked` | players with a position |
| `centre_from_own_goal`, `centre_across` | team centre |
| `spread_length`, `spread_width` | how spread out the team is (std, m) |
| `defensive_line` | 2nd-deepest player (the deepest is the keeper) |
| `front_player` | most advanced player |
| `in_opponent_half` | players in the opponent's half |
| `mean_speed`, `running` | mean speed, players faster than 4 m/s |
| `forward_speed` | mean velocity towards the opponent's goal |
| `marking_distance` | mean distance from each player to the nearest opponent |

Plus `diff_…` = Barça minus Athletic, and `share_walking` (share of players under 2 m/s, from notebook 1).
None of them uses the ball, so the model cannot just copy the clear-control rule.

In [ ]:
X = features.team_features(players_xy, players_speed, slot_team, from_own_goal, forward)
X['share_walking'] = rules.share_walking(players_speed)
example = X.iloc[FRAME]
pd.DataFrame({'Barça': example.filter(like='barca_').values, 'Athletic': example.filter(like='athletic_').values},
             index=[c.removeprefix('barca_') for c in example.filter(like='barca_').index]).round(1)

## 4. Time context

One frame says little; how the teams move over a few seconds says more. Added: 1 s and 5 s
averages of the main columns (`…_1s`, `…_5s`) and how far each team's centre moved up in the
last 2 s (`…_moved_up_2s`).

In [ ]:
X = features.add_time_context(X)
print(X.shape[1], 'player features')
X.loc[FRAME:FRAME + 4, ['barca_forward_speed', 'barca_forward_speed_1s', 'barca_forward_speed_5s', 'barca_moved_up_2s']].round(2)

## 5. Tag features (second model only)

Per Smart Tagging code: +1 while a Barça interval is on, −1 for Athletic, 0 otherwise; plus
`tag_set_piece_on` and `tag_neutral_on`. Kept for a separate model, because `SET PIECES` is also
used by dead-ball rule R4.

In [ ]:
X_tags = features.tag_features(events, len(X))
X_with_tags = pd.concat([X, X_tags], axis=1)
busy = int(np.argmax((X_tags != 0).sum(axis=1).to_numpy() >= 3))  # first frame with 3 tags on
on = X_tags.iloc[busy]
print(f'{X_tags.shape[1]} tag features; on at frame {busy}:', dict(on[on != 0]))

## 6. Train / validation split

Each half (from kickoff) is cut into 2-minute blocks; every 5th is **validation** (V), the rest
**training** (T). Training skips frames within 5 s of a validation block, because the 5 s
averages would see across the border.

Used: labelled frames only, **except** D frames that only R2 / R5 mark (so the model doesn't learn
"ball missing = dead") and the 2nd half before kickoff (D by definition).

In [ ]:
is_val = model.validation_blocks()
val_blocks = runs_of(is_val)
labelled = (labels != UNLABELLED) & graded & ~uncertain_dead
labelled[HALVES[1][1]:SECOND_HALF_KICKOFF] = False
train = labelled & ~model.near(is_val, model.PURGE) & ~is_val
val = labelled & is_val

first_half = range(HALVES[0][1], HALVES[0][2], model.BLOCK)
print('1st half blocks:', ''.join('V' if is_val[f] else 'T' for f in first_half))
pd.DataFrame({name: pd.Series(labels[m]).map(LABEL_NAME).value_counts(normalize=True).round(3)
              for name, m in [(f'train ({train.sum()} frames)', train), (f'validation ({val.sum()} frames)', val)]})

## 7. Two baselines

What a model has to beat:
- **always D**: the most common label;
- **carry-forward**: each validation block gets the last label before it.

In [ ]:
always_d = np.full(len(labels), DEAD, np.int8)
carry_fwd = model.carry_forward(labels, labelled & ~is_val, val_blocks)
results = {'always D': model.scores(labels[val], always_d[val]),
           'carry-forward': model.scores(labels[val], carry_fwd[val])}
pd.DataFrame(results).T.round(3)

## 8. LightGBM: players only, then players + tags

`H vs A` = on frames the rules call H or A, how often the model picks the right team.

In [ ]:
model_players = model.train(X, labels, train)
proba_players = model_players.predict_proba(X)
results['LightGBM players'] = model.scores(labels[val], proba_players[val].argmax(1))

model_tags = model.train(X_with_tags, labels, train)
proba_tags = model_tags.predict_proba(X_with_tags)
results['LightGBM players + tags'] = model.scores(labels[val], proba_tags[val].argmax(1))
pd.DataFrame(results).T.round(3)

Where the players-only model goes wrong (rows = rule label, columns = model, each row sums to 1):

In [ ]:
names = [LABEL_NAME[c] for c in (BARCA, ATHLETIC, DEAD)]
cm = confusion_matrix(labels[val], proba_players[val].argmax(1), labels=[BARCA, ATHLETIC, DEAD], normalize='true')
pd.DataFrame(cm, index=[f'rule {n}' for n in names], columns=[f'model {n}' for n in names]).round(2)

## 9. What the model relies on

Top 10 features by gain (share of the total):

In [ ]:
gain = pd.Series(model_players.booster_.feature_importance('gain'), index=X.columns)
(gain / gain.sum()).sort_values(ascending=False).head(10).round(3)

## 10. Labels where the ball is missing

That is where the model will be used. With the 10 s carry the rules now give some H/A labels there,
so we can finally check the model on them (validation frames, players-only model):

In [ ]:
visible = ~np.isnan(tracking['ball_xy'][:, 0])
pred_val = proba_players.argmax(1)
rows = []
for name, m in [('ball visible', visible), ('ball missing', ~visible)]:
    on_ball = val & m & (labels != DEAD)
    rows.append({'validation frames': name, 'labelled': np.sum(val & m), 'of which H or A': on_ball.sum(),
                 'model right team %': 100 * np.mean(pred_val[on_ball] == labels[on_ball]),
                 'model says D %': 100 * np.mean(pred_val[on_ball] == DEAD)})
pd.DataFrame(rows).round(1)

## 11. Possession level: spellF1 against the rules

Frame scores hide flicker (H H A H H), which spellF1 punishes hard. Reference = rule labels in
each validation block, gaps filled with the previous label. The model's probabilities are averaged
over 1 s or 3 s before taking the most likely label. This is the official metric code, but scored
against our own rules, not the truth.

In [ ]:
reference = model.fill_gaps(labels, val_blocks)
rows = {'carry-forward': carry_fwd, 'players, raw': proba_players.argmax(1).astype(np.int8),
        'players, 1 s': model.smooth(proba_players, FPS), 'players, 3 s': model.smooth(proba_players, 3 * FPS),
        'players + tags, 3 s': model.smooth(proba_tags, 3 * FPS)}
pd.DataFrame([{'prediction': name, 'spellF1': spell_scores(reference, pred, val_blocks)['mean_f1'],
               'possessions': spell_scores(reference, pred, val_blocks)['n_pred']}
              for name, pred in rows.items()]
             + [{'prediction': 'reference', 'spellF1': 1.0,
                 'possessions': spell_scores(reference, reference, val_blocks)['n_gt']}]).round(3)

## 12. Predict the unlabelled frames

Final model: same settings, trained on **all** usable labelled frames (training + validation, no
split needed now). Then its prediction, averaged over 3 s, on:
- the **unlabelled** frames, with the ball visible and missing;
- the **D frames only R2 / R5 mark** (not trained on): does the model agree they are dead?

In [ ]:
final_model = model.train(X, labels, labelled)
proba = final_model.predict_proba(X)
predicted = model.smooth(proba, 3 * FPS)
unlabelled = graded & (labels == UNLABELLED)
groups = [('unlabelled, ball visible', unlabelled & visible), ('unlabelled, ball missing', unlabelled & ~visible),
          ('D only from R2/R5', uncertain_dead)]
pd.DataFrame([{'frames': name, 'n': m.sum(), 'H %': 100 * np.mean(predicted[m] == BARCA),
               'A %': 100 * np.mean(predicted[m] == ATHLETIC), 'D %': 100 * np.mean(predicted[m] == DEAD),
               'mean confidence': proba[m].max(axis=1).mean()} for name, m in groups]).round(2)

### 12a. Gaps with the same team on both sides

An unlabelled gap with the same team before and after it was most likely that team's possession.
How often does the model say so? (With the ball visible the carry rule already fills such gaps, so
only gaps with the ball missing for more than 10 s are left.)

In [ ]:
rows = []
for name, ball in [('ball visible', True), ('ball missing', False)]:
    n = same = other = 0
    for a, b in runs_of(unlabelled):
        before, after = labels[a - 1], labels[b]
        if before == after and before != DEAD and (visible[a:b].mean() > 0.5) == ball:
            n += b - a
            same += np.sum(predicted[a:b] == before)
            other += np.sum((predicted[a:b] != before) & (predicted[a:b] != DEAD))
    if n:
        rows.append({'gap': name, 'frames': n, 'same team %': 100 * same / n, 'other team %': 100 * other / n,
                     'D %': 100 * (n - same - other) / n})
pd.DataFrame(rows).round(1)

### 12b. The whole match, rules + model

Rule labels where they exist, the model's (3 s averaged) prediction elsewhere. No Viterbi yet, so
expect too many short possessions.

In [ ]:
combined = np.where(labels == UNLABELLED, predicted, labels).astype(np.int8)
halves = [(first, last + 1) for _, first, last in HALVES]
spells = [(a, b) for start, end in halves for t in (BARCA, ATHLETIC) for a, b in runs_of(combined[start:end] == t)]
lengths = np.array([b - a for a, b in spells]) / FPS
minutes = graded.sum() / FPS / 60
print(f'H {np.mean(combined[graded] == BARCA):.1%}, A {np.mean(combined[graded] == ATHLETIC):.1%}, '
      f'D {np.mean(combined[graded] == DEAD):.1%}')
print(f'{len(spells)} possessions = {len(spells) / minutes:.1f} per minute; length (s): '
      f'median {np.median(lengths):.1f}, shorter than 1 s: {np.mean(lengths < 1):.0%}')

## Summary

On validation blocks the model never trained on (D frames only from R2/R5 left out):

| | accuracy | macro F1 | H vs A | spellF1 vs rules (3 s) |
|---|---|---|---|---|
| always D | 0.45 | 0.21 | – | – |
| carry-forward | 0.30 | 0.27 | 0.41 | 0.00 |
| **LightGBM players** | **0.79** | **0.78** | **0.77** | **0.43** |
| LightGBM players + tags | 0.79 | 0.78 | 0.78 | 0.42 |

- **Player movement alone carries real possession signal.** Strongest H/A cue: **team width**
  (the team with the ball spreads out). Tags add nothing.
- **Ball missing, live play (section 10):** right team 58%, **D 40%**, wrong team only 2%. So when the
  model picks a team it is right ~96% of the time; its problem is calling live play dead.
- **The fix did not remove that bias:** on unlabelled ball-missing frames it still says D 66%
  (64% before the longer carry and without R2/R5 D in training). It also says D on 90% of the
  R2/R5-only frames it never trained on: players there move like in the trusted stoppages.
- **Rules + model over the whole match (12b):** D 48% (high), 2.8 possessions per minute, median 7.2 s.
- **It flickers:** 163 possessions raw vs 57 in the reference; 3 s averaging brings it to 55.

**Next:** let the rules decide live vs dead, and use the model only for H vs A (its two
probabilities, ignoring D) on frames the rules don't call dead; then Viterbi.